## HateXplain - 7 OBJECTIVES

### Multi Task Learning

* **Dataset:** HateXplain
* **Task :** Classification

* **Target 1:** Hate speech against Women
* **Target 2:** Hate speech against Men
* **Target 3:** Hate speech against Homosexual
* **Target 4:** Hate speech against Indigenous
* **Target 5:** Hate speech against African
* **Target 6:** Hate speech against Asian
* **Target 7:** Hate speech against Hispanic
* **Target 8:** Hate speech against Christian
* **Target 9:** Hate speech against Hindu
* **Target 10:** Hate speech against Islam

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from datasets import load_dataset

In [ ]:
from machinemoo import moo
from machinemoo import get_objectives, get_models
from machinemoo.analysis.visualization import plot_pareto, plot_multiple_hypervolumes
from machinemoo.analysis.metrics import compute_hypervolume_progress

import preprocessing as pp
from scalarization import NLPScalarization

In [ ]:
# Setting a seed for reproducibility
from machinemoo.utils.seed_config import set_np_torch_seed
set_np_torch_seed(42)

In [ ]:
import pickle

In [ ]:
PREFIX_SAVE_IMAGE = 'images/10_objs'

In [ ]:
PREFIX_SAVE_RESULT = '10_objs'

## Dataset and Model

In [ ]:
# Get HateXplain Dataset
dataset = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)

# Choose BERT model
model_name="bert-base-uncased"

In [ ]:
# Use GPU if its available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
task_config = {
    "Women": {"type": "group_hate", "group": "Women"},
    "Men": {"type": "group_hate", "group": "Men"},
    "Homosexual": {"type": "group_hate", "group": "Homosexual"},
    "Indigenous": {"type": "group_hate", "group": "Indigenous"},
    "African": {"type": "group_hate", "group": "African"},
    "Asian": {"type": "group_hate", "group": "Asian"},
    "Jewish": {"type": "group_hate", "group": "Jewish"},
    #"Caucasian": {"type": "group_hate", "group": "Caucasian"},
    "Hispanic": {"type": "group_hate", "group": "Hispanic"},
    #"Buddhism": {"type": "group_hate", "group": "Buddhism"},
    #"Christian": {"type": "group_hate", "group": "Christian"},
    #"Hindu": {"type": "group_hate", "group": "Hindu"},
    "Islam": {"type": "group_hate", "group": "Islam"},
    #"Heteresexual": {"type": "group_hate", "group": "Heteresexual"},
    
}

train_dataloader = pp.get_dataloader(dataset['train'], task_config)

## MOO Modeling and Training

* MOO methods:
    * MOLA
    * Random Weights

In [ ]:
# Setting optimization parameter to use in all methods in this experiment
opt_params = {
    'node_time_limit': 2,
    'target_size': 500,
    'target_gap': 0,
    'node_gap': 0.05,
    'norm': False
}
results = {}
objectives_names = list(task_config.keys())

# Control optimization output (Default is False):
#   verbose=True -> show optimization progress
#   debug=True   -> show detailed debug information
verbose = False
debug = False

def machine_optimize(method, lb_estimate=0.01):
    w_scalar = NLPScalarization(
        train_dataloader = train_dataloader, 
        device = device, 
        task_names = objectives_names, 
        lower_bound_estimate=0.01
    )
    moopt = moo(w_scalar, verbose=verbose, debug=debug).mo_optimization(method, **opt_params)
    objs = get_objectives(moopt)
    if method == 'mola':
        hypervolume_values = moopt.get_hypervolumes()
    else:
        hypervolume_values = compute_hypervolume_progress(objs)
    if lb_estimate == 'lipschitz':
        method = f'{method}_grad'
    results[method] = {
                    "moopt": moopt,
                    "objectives": objs,
                    "models": get_models(moopt),
                    "hypervolume": hypervolume_values
                }
    
def plot_frontier(method, color="#d73027"):
    title="Loss Trade-offs in Multi-Group Hate Speech Detection"
    plot_pareto(methods={method: results[method]['objectives']},
                labels=objectives_names, 
                title=title,
                color=color,
                fontsize=20,
                figsize=(10,7),
                save_path=f'{PREFIX_SAVE_IMAGE}_{method}.pdf'
                )

### MOLA

In [ ]:
method = 'mola'
machine_optimize(method=method)

In [ ]:
plot_frontier(method, color='#d73027')

In [ ]:
# save results
import dill
with open(f"results_{PREFIX_SAVE_RESULT}.pkl", "wb") as f:
    dill.dump(results, f)

### MOLA with gradient

In [ ]:
method = 'mola'
machine_optimize(method=method, lb_estimate='lipschitz')

In [ ]:
plot_frontier(f'{method}_grad', color='#fc8d59')

In [ ]:
# save results
with open(f"results_{PREFIX_SAVE_RESULT}.pkl", "wb") as f:
    dill.dump(results, f)

### Random Weight

In [ ]:
method = 'random_weights'
machine_optimize(method=method)

### Non-dominated Solutions Analysis

In [ ]:
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
rw_solutions = results['random_weights']['objectives']
nd_indices = NonDominatedSorting().do(rw_solutions, only_non_dominated_front=True)
rw_non_dominated_solutions = rw_solutions[nd_indices]

print(f'Number of non-dominated solutions with MOLA          : {len(results['mola']['objectives'])}')
print(f'Number of non-dominated solutions with MOLA GRAD     : {len(results['mola_grad']['objectives'])}')
print(f'Number of non-dominated solutions with RANDOM WEIGHT : {len(rw_non_dominated_solutions)}')

## A posteriori Decision Making

* Comparison between MOO methods
* Ensemble and Evaluation

In [ ]:
pareto_dict = {
        method: res['objectives']
        for method, res in results.items()
        if 'objectives' in res
    }

pareto_dict['random_weights'] = rw_non_dominated_solutions

hypervolumes_dict = {
        method: res['hypervolume']
        for method, res in results.items()
        if 'hypervolume' in res
    }

plot_pareto(methods=pareto_dict,
            labels=objectives_names, 
            #title='Comparative Pareto Frontiers Across MOO Methods',
            #subset=equal.objs,
            fontsize=23,
            figsize=(10,7),
            save_path=f'{PREFIX_SAVE_IMAGE}_methods.pdf'
            )

plot_multiple_hypervolumes(methods_hv=hypervolumes_dict, save_path=f'{PREFIX_SAVE_IMAGE}_hypervolume.pdf')

In [ ]:
# save results
with open(f"results_{PREFIX_SAVE_RESULT}.pkl", "wb") as f:
    dill.dump(results, f)

# get saved results
#with open(f"results_{PREFIX_SAVE_RESULT}.pkl", "rb") as f:
#    outro_dict = pickle.load(f)